# Advanced Time Series Forecasting, Optimization and Explainability

## Final Project — Advanced Topics in Deep Learning (2025/2026)

### Developed by:

- António Cruz (140129)
- Cátia Brás (120093)
- Ricardo Kyaseller (95813)

## 1. Introduction and Project Goals

This project builds upon the Time Series Modelling mini-project to develop a systematic, optimized, and well-analysed forecasting system for air temperature prediction using the Jena Climate dataset. While the mini-project focused on exploratory modelling and understanding different design choices, this Final Project elevates performance to a primary objective, achieved through systematic optimization and supported by rigorous analysis.

The core forecasting task is a **multivariate-input, univariate-output, multi-step prediction** problem: given a window of 120 hours of historical meteorological measurements (temperature, pressure, humidity, wind speed, maximum wind speed, and wind direction), the system predicts the air temperature for the next 24 hours.

Our pipeline integrates the following advanced techniques:

- **Baseline Deep Learning Model (GRU):** A configurable multi-layer GRU architecture serves as the reference model against which all improvements are measured. The baseline incorporates regularization (L2, dropout, Gaussian noise), gradient clipping, and modern training strategies (AdamW optimizer, Huber loss).

- **Evolutionary Optimization:** A genetic algorithm (GA) automatically searches over an end-to-end pipeline configuration space, jointly optimizing model hyperparameters, architectural choices (number of layers, units, activation functions), training settings (optimizer, learning rate, loss function, batch size), preprocessing strategy (scaler type), and regularization. This goes beyond isolated hyperparameter tuning by treating the full pipeline as the unit of optimization.

- **Synthetic Data Generation (TimeGAN):** A Time-series Generative Adversarial Network generates realistic synthetic weather sequences from the training set, enabling data augmentation without information leakage. The impact on forecasting robustness and generalization is evaluated by comparing models trained with and without synthetic data.

- **Explainable AI (XAI):** Both global and local explanation methods are applied to understand which input features and temporal patterns drive the model's predictions, providing interpretability and trust in the forecasting system.

- **Efficiency and Resource Analysis:** Training time, inference latency, parameter counts, and memory usage are profiled across model configurations, enabling an informed analysis of accuracy–efficiency trade-offs.

The remainder of this report is organized as follows: Section 2 describes the experimental setup and reproducibility measures; Section 3 presents the dataset and problem formulation; Sections 4–6 cover data preparation, feature engineering, and the windowing pipeline; Section 7 establishes the baseline models; Section 8 details the TimeGAN synthetic data generation; Section 9 presents the evolutionary optimization process; Section 10 covers final model retraining and selection; Sections 11–12 present the explainability and efficiency analyses; and Sections 13–14 provide the comparative discussion and conclusion.

## 2. Environment, Reproducibility and Experimental Setup

### 2.1 Libraries Import

The following cell imports all libraries used throughout the notebook. Core dependencies include TensorFlow/Keras for deep learning, NumPy/Pandas for data manipulation, Matplotlib for visualization, and scikit-learn for evaluation metrics.

The project is implemented in Python 3.12 using the following key libraries:

| Library | Purpose |
|---------|---------|
| TensorFlow / Keras | Deep learning framework for GRU models and TimeGAN |
| NumPy, Pandas | Data manipulation and numerical computation |
| scikit-learn | Scalers (StandardScaler, RobustScaler, MinMaxScaler) and evaluation metrics |
| Matplotlib, Seaborn | Visualization |
| psutil | System-level memory and resource monitoring |

All dependencies are specified in `requirements.txt` for reproducibility.

In [ ]:
import os
import sys
import random
import warnings
from pathlib import Path

# Suppress TensorFlow informational logs and oneDNN messages
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings("ignore")

### 2.2 Reproducibility

To ensure reproducible results across runs, we set a global random seed (`SEED = 42`) applied consistently to:

- Python's built-in `random` module,
- NumPy's random number generator,
- TensorFlow's global and operation-level random seeds.

This is handled by the utility function `set_global_seed()` from `src.utils.env`. While perfect determinism on GPU is not guaranteed due to non-deterministic reduction operations, fixing all controllable seeds ensures that results are reproducible to a high degree.

### 2.3 GPU Configuration

TensorFlow's GPU memory growth is enabled via `enable_gpu_memory_growth()`, which allows the framework to allocate memory incrementally rather than reserving the full GPU memory upfront. This prevents out-of-memory errors when running multiple models (as required during evolutionary search) and enables more efficient resource sharing.

### 2.4 Project Structure

The codebase follows a modular design, separating concerns across distinct packages:

```
src/
├── features/       # Feature engineering, scaling, windowing
├── models/         # GRU model builder and training utilities
├── evolution/      # Evolutionary search: search space, operators, fitness
├── gan/            # TimeGAN architecture, config, data preparation
└── utils/          # Environment setup, profiling tools
```

This modular approach allows the notebook to focus on experimental orchestration and analysis while keeping reusable logic in tested, importable modules.

In [ ]:
SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

gpus = tf.config.list_physical_devices("GPU")
print("GPUs:", gpus)

for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except Exception:
        pass

In [ ]:
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print(PROJECT_ROOT)

In [ ]:
from src.utils.env import set_global_seed, enable_gpu_memory_growth, get_device_info

SEED = 42
set_global_seed(SEED)
enable_gpu_memory_growth()

print(get_device_info())

## 3. Dataset and Problem Definition

### 3.1 Dataset Description

The Jena Climate dataset contains meteorological measurements collected at a weather station in Jena, Germany, between January 2009 and December 2016. The raw data is sampled every 10 minutes, yielding approximately 420,000 observations across 15 meteorological variables.

The dataset is publicly available at: https://keras.io/examples/timeseries/timeseries_weather_forecasting

In [ ]:
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "src" / "data"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("DATA_DIR exists?", DATA_DIR.exists())

In [ ]:
DATASET_PATH = DATA_DIR / "jena_climate_2009_2016.csv"

df = pd.read_csv(DATASET_PATH)
df.head()

### 3.2 Initial Inspection and Quality Assurance

Upon loading, we performed the following quality checks:

- **Duplicate detection:** Verified that no duplicate timestamps exist in the raw data.
- **Missing value analysis:** Checked for NaN entries across all columns.
- **Temporal ordering:** Confirmed that the dataset is strictly ordered by time, with no gaps or inversions in the datetime index.
- **Sampling regularity:** Examined the distribution of time deltas between consecutive observations to detect any irregularities in the 10-minute sampling frequency.

In [ ]:
print("Dataset path:", DATASET_PATH)
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
print("\nInfo:")
df.info()

After loading the raw dataset, we perform a series of quality checks to ensure data integrity before any transformations. Specifically, we verify the presence of duplicated rows and inspect all columns for missing values. Any duplicated observations are removed to avoid biasing the resampling step that follows. This is a standard practice in time series preprocessing — duplicate timestamps can arise from logging artifacts and must be eliminated to ensure a consistent temporal index.

In [ ]:
print("Duplicated rows:", df.duplicated().sum())
print("\nMissing values per column:")
print(df.isna().sum())

In [ ]:
df = df.drop_duplicates().copy()

print("Shape after removing duplicates:", df.shape)
print("Duplicated rows after cleanup:", df.duplicated().sum())

### 3.3 Temporal Resampling

The raw 10-minute resolution was resampled to **1-hour intervals** using mean aggregation. This decision reduces the dataset size by a factor of 6 (from ~420k to ~70k observations) while preserving the dominant meteorological patterns, which operate on hourly-to-daily timescales. Hourly resolution also aligns the lookback and horizon parameters with intuitive units (hours), making the forecasting task more interpretable.

After resampling, any rows with NaN values (resulting from incomplete hourly bins at dataset boundaries) were dropped.

In [ ]:
df["Date Time"] = pd.to_datetime(df["Date Time"], dayfirst=True)
df = df.sort_values("Date Time").reset_index(drop=True)

print(df["Date Time"].min(), "->", df["Date Time"].max())
df[["Date Time"]].head()

In [ ]:
df["Date Time"].diff().value_counts().head(10)

### 3.4 Hourly Resampling

The actual resampling operation uses Pandas' `resample("1h").mean()` on the datetime-indexed dataframe, aggregating all 10-minute observations within each hourly bin into their arithmetic mean. This is applied uniformly across all meteorological variables. Mean aggregation was chosen over alternatives (e.g., first/last value, median) because it preserves the central tendency of each hour's measurements while smoothing out sub-hourly noise that is irrelevant for the 24-hour forecasting task.

In [ ]:
df_hourly = (
    df.set_index("Date Time")
      .resample("1h")
      .mean()
      .reset_index()
)

print("Original shape:", df.shape)
print("Hourly shape:", df_hourly.shape)
print(df_hourly.head())

### 3.5 Post-Resampling Quality Check

After resampling, we re-validate the dataset to ensure the operation did not introduce artifacts. We verify that no duplicate hourly timestamps exist, that the time delta between consecutive rows is consistently 1 hour, and that any missing values (from incomplete hourly bins at the boundaries of data gaps) are identified and quantified. This step is critical because the sliding window approach used later assumes a regular, gap-free time series.

In [ ]:
print("Missing values after hourly resampling:")
print(df_hourly.isna().sum())

print("\nDuplicated timestamps after resampling:", df_hourly["Date Time"].duplicated().sum())

print("\nTime step distribution after resampling:")
print(df_hourly["Date Time"].diff().value_counts().head(10))

### 3.6 Handling Missing Values After Resampling

Any rows containing NaN values after resampling are dropped. Since the original dataset has near-complete 10-minute coverage across the 2009–2016 period, the number of affected rows is minimal (88 out of ~70,000) and does not introduce meaningful temporal gaps. Dropping these rows is preferred over imputation because the affected bins are concentrated at dataset boundaries and isolated gaps, where interpolation would risk introducing artificial patterns.

In [ ]:
df_hourly = df_hourly.dropna().reset_index(drop=True)

print("Shape after dropping NaNs:", df_hourly.shape)
print("\nMissing values after cleanup:")
print(df_hourly.isna().sum().sum())
print("\nTime step distribution after cleanup:")
print(df_hourly["Date Time"].diff().value_counts().head())

### 3.7 Forecasting Task and Variable Selection

Following the project specification, we retain 6 meteorological variables plus the datetime reference: `T (degC)` (air temperature, the target), `p (mbar)` (atmospheric pressure), `rh (%)` (relative humidity), `wv (m/s)` (wind speed), `max. wv (m/s)` (maximum wind speed), and `wd (deg)` (wind direction). All other variables from the original 15-column dataset are excluded to maintain a focused and non-trivial forecasting problem.

This defines a **multivariate-input, univariate-output, multi-step** forecasting problem: the model receives a 5-day window of multi-sensor weather data (120 hourly time steps across all features) and must predict the air temperature trajectory for the next 24 hours. The multivariate input ensures the model can leverage cross-variable dependencies — for instance, the relationship between falling pressure and subsequent temperature changes driven by weather fronts.

In [ ]:
selected_columns = [
    "Date Time",
    "T (degC)",
    "p (mbar)",
    "rh (%)",
    "wv (m/s)",
    "max. wv (m/s)",
    "wd (deg)"
]

df_model = df_hourly[selected_columns].copy()

print("Selected columns:")
print(df_model.columns.tolist())
print("\nShape:", df_model.shape)
df_model.head()

## 4. Data Initial Preparation

### 4.1 Target Definition and Base Modeling DataFrame

The target variable is defined as `T (degC)` — air temperature in degrees Celsius. This is the only variable the model must predict in the output window. The `Date Time` column serves as the temporal reference for feature engineering (extracting hour-of-day and day-of-year) but is not used directly as a model input. The remaining meteorological variables (`p`, `rh`, `wv`, `max. wv`, `wd`) form the exogenous covariates that provide atmospheric context for the temperature forecast.

In [ ]:
TARGET_COL = "T (degC)"
TIME_COL = "Date Time"

feature_cols = [col for col in df_model.columns if col not in [TIME_COL]]
input_feature_cols = [col for col in feature_cols]

print("Target:", TARGET_COL)
print("Time column:", TIME_COL)
print("Input features:", input_feature_cols)
print("Number of input features:", len(input_feature_cols))

### 4.2 Base Forecasting Data Overview

We inspect the prepared dataframe to confirm shape, date range, and column integrity before proceeding to feature engineering. The dataset at this stage contains 70,041 hourly observations spanning from January 2009 to January 2017, with 7 columns (datetime + 6 meteorological variables). This verification step ensures that no data was inadvertently lost during the resampling and cleaning process.

In [ ]:
print(df_model.head())
print("\nShape:", df_model.shape)
print("\nDate range:", df_model[TIME_COL].min(), "->", df_model[TIME_COL].max())

## 5. Feature Engineering

Two families of derived features are introduced to enrich the input representation: **cyclical temporal encodings** and **wind-derived features**. The goal is to provide the neural network with representations that respect the physical structure of the data — cyclic quantities are encoded cyclically, and vector quantities are decomposed into Cartesian components. All feature engineering is implemented in `src.features.engineering` for reproducibility.

In [ ]:
from src.features.engineering import (
    add_time_features,
    add_wind_features,
    get_final_feature_columns,
)
from src.features.windowing import make_windows
from src.models.gru import build_gru_model

### 5.1 Cyclical Time Features

Hour-of-day and day-of-year carry strong periodic signals (diurnal and seasonal cycles). Encoding them as raw integers would introduce artificial discontinuities (e.g., hour 23 far from hour 0). We apply a sine–cosine encoding that maps each cyclic variable to a point on the unit circle, ensuring smooth transitions at boundaries:
- `hour_sin`, `hour_cos` — diurnal cycle (period = 24h)
- `doy_sin`, `doy_cos` — seasonal cycle (period = 365.25 days)

In [ ]:
df_feat = add_time_features(df_model, time_col=TIME_COL)

df_feat[[
    TIME_COL, "hour", "dayofyear",
    "hour_sin", "hour_cos", "doy_sin", "doy_cos"
]].head()

### 5.2 Wind-Derived Features

Wind direction in degrees is inherently circular and difficult for neural networks to interpret directly. We derive six features:
- `wd_sin`, `wd_cos` — cyclic encoding of wind direction
- `wx`, `wy` — Cartesian decomposition of wind vector (speed × cos/sin of direction), replacing the polar representation with a form neural networks process more naturally
- `wind_gap` — difference between max and sustained wind speed (gust intensity)
- `gust_ratio` — ratio of max to sustained wind speed (relative gust strength, with ε = 10⁻⁶ for numerical stability)

In [ ]:
df_feat = add_wind_features(df_feat)

df_feat[[
    TIME_COL, "wv (m/s)", "max. wv (m/s)", "wd (deg)",
    "wd_sin", "wd_cos", "wx", "wy", "wind_gap", "gust_ratio"
]].head()

### 5.3 Final Feature Set for Modeling

The complete input feature vector contains **16 dimensions**: 6 original meteorological variables + 4 cyclical temporal encodings + 6 wind-derived features. This feature set is defined in `get_final_feature_columns()` and used consistently across all experiments — baseline, evolutionary optimization, and synthetic data generation. The expansion from 6 to 16 features provides the model with richer representations without introducing external data sources or violating the variable selection constraints of the project specification.

In [ ]:
final_feature_cols = get_final_feature_columns()

print("Number of modeling features:", len(final_feature_cols))
print(final_feature_cols)

## 6. Split, Scaling and Windowing

### 6.1 Temporal Train/Validation/Test Split

The dataset is split chronologically (70% / 15% / 15%) with **no shuffling** — the split respects temporal order to prevent data leakage. The training set covers the earliest years, followed by validation, with the most recent data reserved for testing. This mirrors a realistic deployment scenario.

In [ ]:
n = len(df_feat)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

df_train = df_feat.iloc[:train_end].copy()
df_val = df_feat.iloc[train_end:val_end].copy()
df_test = df_feat.iloc[val_end:].copy()

print("Train shape:", df_train.shape)
print("Validation shape:", df_val.shape)
print("Test shape:", df_test.shape)

### 6.2 Feature Scaling

All input features are normalized using **StandardScaler** (zero mean, unit variance) as the baseline strategy. The scaler is fit exclusively on the training set and then applied via `transform` to the validation and test sets, preventing any information leakage from future data into the scaling parameters. This is a critical design choice — fitting the scaler on the full dataset would allow statistics from the validation and test periods to influence the training data representation, violating the temporal split.

The evolutionary optimization also searches over the scaler type, considering StandardScaler, RobustScaler (median/IQR-based, robust to outliers), and MinMaxScaler (scales to [0, 1]). This allows the GA to discover whether a different normalization strategy benefits specific model configurations.

In [ ]:
from src.features.scaling import get_scaler

In [ ]:
SCALER_NAME = "standard"   # baseline oficial; no EA poderá ser standard/robust/minmax
scaler = get_scaler(SCALER_NAME)

X_train_df = df_train[final_feature_cols].copy()
X_val_df = df_val[final_feature_cols].copy()
X_test_df = df_test[final_feature_cols].copy()

X_train_scaled = scaler.fit_transform(X_train_df)
X_val_scaled = scaler.transform(X_val_df)
X_test_scaled = scaler.transform(X_test_df)

print("Scaler:", SCALER_NAME)
print("Scaled train shape:", X_train_scaled.shape)
print("Scaled val shape:", X_val_scaled.shape)
print("Scaled test shape:", X_test_scaled.shape)

### 6.3 Target Index for Forecasting

We identify the position of `T (degC)` within the 16-feature vector. This index is used during the windowing step to extract only the temperature column for the output sequence **y**, while keeping all 16 features in the input tensor **X**. This separation is what makes the problem multivariate-input but univariate-output: the model sees all meteorological context as input but is only evaluated on its ability to predict future temperature.

In [ ]:
target_idx = final_feature_cols.index(TARGET_COL)

print("Target column:", TARGET_COL)
print("Target index:", target_idx)

### 6.4 Supervised Windowing

The scaled data is converted into supervised learning samples using a sliding window:
- **Input (X):** `(LOOKBACK, 16)` = 120 consecutive hourly observations across all features (5 days of context)
- **Output (y):** `(HORIZON,)` = 24 hourly temperature values to predict (1 day ahead)

The window slides by one time step, generating the maximum number of training samples.

In [ ]:
from src.features.windowing import make_windows

In [ ]:
LOOKBACK = 120
HORIZON = 24

X_train, y_train = make_windows(X_train_scaled, target_idx, LOOKBACK, HORIZON)
X_val, y_val = make_windows(X_val_scaled, target_idx, LOOKBACK, HORIZON)
X_test, y_test = make_windows(X_test_scaled, target_idx, LOOKBACK, HORIZON)

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val:", X_val.shape, "y_val:", y_val.shape)
print("X_test:", X_test.shape, "y_test:", y_test.shape)

### 6.5 Windowed Data Sanity Check

We verify the shapes of all windowed arrays to confirm correct dimensionality and consistency between input and output samples across train, validation, and test splits. The expected shapes are: input `(N, 120, 16)` — N samples, each with 120 hourly time steps across 16 features; output `(N, 24)` — N corresponding 24-hour temperature forecast targets. This verification catches common off-by-one errors in window construction that would silently corrupt the training process.

In [ ]:
print("Input shape:", X_train.shape[1:])
print("Forecast horizon:", y_train.shape[1])
print("Number of input features:", X_train.shape[2])

print("\nExample input window shape:", X_train[0].shape)
print("Example target shape:", y_train[0].shape)
print("First 5 target values (scaled):", y_train[0][:5])

## 7. Baseline Models

Two baseline models are established as reference points for evaluating all subsequent improvements. The **persistence baseline** provides the lowest bar — the performance of a no-skill forecast that any learned model must surpass. The **GRU baseline** represents a carefully designed deep learning model with modern training practices, serving as the benchmark against which the evolutionary optimization is measured.

### 7.1 Persistence Baseline

The simplest possible forecast: repeat the last observed temperature value across the entire 24-hour horizon. This naive baseline provides a scale-aware reference — any learned model that fails to beat persistence adds no value. We first build a simple GRU as a quick sanity check, then proceed to the official configurable baseline in Section 7.2.

In [ ]:
def persistence_forecast(X):
    # Repeats the last observed target value across the full forecast horizon
    last_temp = X[:, -1, target_idx]
    return np.repeat(last_temp[:, None], HORIZON, axis=1)

y_pred_persistence = persistence_forecast(X_test)

print("Persistence prediction shape:", y_pred_persistence.shape)
print("First prediction:", y_pred_persistence[0][:5])

In [ ]:
persistence_mae_scaled = mean_absolute_error(y_test.flatten(), y_pred_persistence.flatten())
persistence_rmse_scaled = np.sqrt(mean_squared_error(y_test.flatten(), y_pred_persistence.flatten()))

print("Persistence MAE (scaled):", persistence_mae_scaled)
print("Persistence RMSE (scaled):", persistence_rmse_scaled)

In [ ]:
def build_gru_baseline(input_shape, horizon):
    model = keras.Sequential([
        layers.Input(shape=input_shape),
        layers.GRU(64, return_sequences=False),
        layers.Dropout(0.2),
        layers.Dense(64, activation="relu"),
        layers.Dense(horizon)
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="mse",
        metrics=["mae"]
    )
    return model

gru_baseline = build_gru_baseline(
    input_shape=(X_train.shape[1], X_train.shape[2]),
    horizon=HORIZON
)

gru_baseline.summary()

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

history_baseline = gru_baseline.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=64,
    callbacks=[early_stopping],
    verbose=1
)

In [ ]:
y_pred_gru = gru_baseline.predict(X_test, verbose=0)

print("Prediction shape:", y_pred_gru.shape)
print("First prediction:", y_pred_gru[0][:5])

In [ ]:
gru_mae_scaled = mean_absolute_error(y_test.flatten(), y_pred_gru.flatten())
gru_rmse_scaled = np.sqrt(mean_squared_error(y_test.flatten(), y_pred_gru.flatten()))

print("GRU MAE (scaled):", gru_mae_scaled)
print("GRU RMSE (scaled):", gru_rmse_scaled)

### 7.2 GRU Baseline (Official)

The official baseline uses `build_gru_model()` from `src.models.gru` — a configurable 2-layer GRU with AdamW optimizer, Huber loss (δ=1.0), L2 regularization, gradient clipping, and an intermediate dense layer. Training uses early stopping (patience=6) and learning rate reduction (patience=3, factor=0.5) via `train_model()` from `src.models.train_eval`.

In [ ]:
from src.models.gru import build_gru_model

In [ ]:
BASELINE_CFG = {
    "n_layers": 2,
    "units1": 96,
    "units2": 64,
    "units3": 96,
    "dropout": 0.0,
    "l2": 1e-6,
    "dense_units": 256,
    "dense_activation": "relu",
    "learning_rate": 2e-4,
    "clipnorm": 2.0,
    "optimizer_name": "adamw",
    "weight_decay": 1e-6,
    "loss_name": "huber1",
    "gaussian_noise_std": 0.0,
    "batch_size": 128,
}

gru_baseline = build_gru_model(
    L=LOOKBACK,
    n_features=X_train.shape[2],
    H=HORIZON,
    units1=BASELINE_CFG["units1"],
    units2=BASELINE_CFG["units2"],
    units3=BASELINE_CFG["units3"],
    n_layers=BASELINE_CFG["n_layers"],
    dropout=BASELINE_CFG["dropout"],
    l2=BASELINE_CFG["l2"],
    dense_units=BASELINE_CFG["dense_units"],
    dense_activation=BASELINE_CFG["dense_activation"],
    learning_rate=BASELINE_CFG["learning_rate"],
    clipnorm=BASELINE_CFG["clipnorm"],
    optimizer_name=BASELINE_CFG["optimizer_name"],
    weight_decay=BASELINE_CFG["weight_decay"],
    loss_name=BASELINE_CFG["loss_name"],
    gaussian_noise_std=BASELINE_CFG["gaussian_noise_std"],
)

gru_baseline.summary()

In [ ]:
from src.models.train_eval import (
    train_model,
    evaluate_scaled_forecasts,
    inverse_scale_target,
    evaluate_original_scale_forecasts,
)

In [ ]:
history_baseline = train_model(
    model=gru_baseline,
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    batch_size=BASELINE_CFG["batch_size"],
    epochs=60,
    verbose=1
)

In [ ]:
y_pred_gru = gru_baseline.predict(X_test, verbose=0)

print("Prediction shape:", y_pred_gru.shape)
print("First prediction:", y_pred_gru[0][:12])

In [ ]:
gru_scaled_metrics = evaluate_scaled_forecasts(y_test, y_pred_gru)

gru_mae_scaled = gru_scaled_metrics["mae_scaled"]
gru_rmse_scaled = gru_scaled_metrics["rmse_scaled"]

print("GRU Baseline MAE (scaled):", gru_mae_scaled)
print("GRU Baseline RMSE (scaled):", gru_rmse_scaled)

### 7.3 Baseline Comparison (Scaled)

We present a side-by-side comparison of the persistence and GRU baseline models on scaled metrics (MAE and RMSE computed on standardized data). Scaled metrics are useful for monitoring training dynamics and comparing models that use the same scaler, but they lack direct physical interpretation — a scaled MAE of 0.19 does not convey how many degrees Celsius the model is off by. For this reason, we also compute original-scale metrics in the following subsections.

In [ ]:
baseline_results_scaled = pd.DataFrame({
    "Model": ["Persistence", "GRU Baseline Official"],
    "MAE_scaled": [persistence_mae_scaled, gru_mae_scaled],
    "RMSE_scaled": [persistence_rmse_scaled, gru_rmse_scaled],
})

baseline_results_scaled

### 7.4 Reverse Scaling

Predictions and ground truth are inverse-transformed from normalized space back to degrees Celsius using the training set's scaler statistics (mean and standard deviation of the temperature column). This allows evaluation in physically meaningful units.

In [ ]:
target_mean = scaler.mean_[target_idx]
target_std = scaler.scale_[target_idx]

y_test_inv = inverse_scale_target(y_test, target_mean, target_std)
y_pred_persistence_inv = inverse_scale_target(y_pred_persistence, target_mean, target_std)
y_pred_gru_inv = inverse_scale_target(y_pred_gru, target_mean, target_std)

print("y_test_inv shape:", y_test_inv.shape)
print("y_pred_gru_inv shape:", y_pred_gru_inv.shape)

### 7.5 Baseline Evaluation in Original Temperature Scale

We compute MAE and RMSE in degrees Celsius on the inverse-scaled predictions. These original-scale metrics are the primary comparison basis throughout the report, as they have direct physical interpretation: an MAE of 1.67°C means the model's 24-hour temperature forecasts are off by an average of 1.67 degrees. This is the metric used as the evolutionary fitness function and the final evaluation criterion.

In [ ]:
gru_original_metrics = evaluate_original_scale_forecasts(y_test_inv, y_pred_gru_inv)
persistence_original_metrics = evaluate_original_scale_forecasts(y_test_inv, y_pred_persistence_inv)

gru_mae = gru_original_metrics["mae"]
gru_rmse = gru_original_metrics["rmse"]

persistence_mae = persistence_original_metrics["mae"]
persistence_rmse = persistence_original_metrics["rmse"]

print("GRU Baseline MAE (°C):", gru_mae)
print("GRU Baseline RMSE (°C):", gru_rmse)

### 7.6 Final Baseline Comparison

The combined comparison table presents both scaled and original-scale (°C) metrics for the persistence and GRU baseline models. The GRU baseline achieves a substantial improvement over the naive persistence forecast, confirming that the recurrent architecture successfully captures temporal dependencies in the meteorological data. These results establish the performance floor and ceiling that the evolutionary optimization will attempt to improve upon.

In [ ]:
baseline_results = pd.DataFrame({
    "Model": ["Persistence", "GRU Baseline Official"],
    "MAE_scaled": [persistence_mae_scaled, gru_mae_scaled],
    "RMSE_scaled": [persistence_rmse_scaled, gru_rmse_scaled],
    "MAE_degC": [persistence_mae, gru_mae],
    "RMSE_degC": [persistence_rmse, gru_rmse],
})

baseline_results

### 7.7 Example Forecast Visualization

Visual comparison of a single 24-hour forecast window: ground truth vs. persistence vs. GRU baseline predictions in °C. This provides an intuitive sense of how the models track the temperature trajectory.

In [ ]:
sample_idx = 0

plt.figure(figsize=(10, 5))
plt.plot(y_test_inv[sample_idx], label="True", marker="o")
plt.plot(y_pred_persistence_inv[sample_idx], label="Persistence", linestyle="--")
plt.plot(y_pred_gru_inv[sample_idx], label="GRU Baseline", linestyle="--")
plt.title("Example Multi-step Forecast on Test Set")
plt.xlabel("Forecast Step")
plt.ylabel("Temperature (°C)")
plt.legend()
plt.grid(True)
plt.show()

## 8. Synthetic Data Generation with TimeGAN

### 8.1 Motivation and Experimental Role

The goal of the TimeGAN component is to generate realistic synthetic multivariate weather sequences using only the training split, avoiding any form of data leakage. 

These synthetic sequences will later be used to augment the training data and assess whether data augmentation improves forecasting performance, robustness, or generalization.

To keep the setup consistent with the forecasting task, the synthetic generator is trained on multivariate sequences of length `LOOKBACK + HORIZON`, so that each generated sequence can later be split into:
- an input window of length `LOOKBACK`
- a target forecasting horizon of length `HORIZON`

The impact of synthetic augmentation will be evaluated by comparing forecasting models trained with:
1. real data only
2. real data plus synthetic data

### 8.2 TimeGAN Training Data Preparation

Training data is converted into fixed-length sequences of `LOOKBACK + HORIZON = 144` time steps, matching the forecasting window. This design choice ensures that each generated synthetic sequence can be directly split into an input window (120 steps) and a target window (24 steps) compatible with the forecasting pipeline, without any additional post-processing.

Only the **training set** is used for sequence extraction — validation and test data are never exposed to the generative model. This is essential to prevent data leakage: if the TimeGAN learned distributional characteristics from future data, any augmentation benefit would be artificially inflated.

In [ ]:
from src.gan.data_prep import make_timegan_sequences, split_synthetic_sequences

TIMEGAN_SEQ_LEN = LOOKBACK + HORIZON

timegan_train_sequences = make_timegan_sequences(X_train_scaled, TIMEGAN_SEQ_LEN)

print("TimeGAN sequence length:", TIMEGAN_SEQ_LEN)
print("TimeGAN training sequences shape:", timegan_train_sequences.shape)

### 8.3 TimeGAN Model and Training Procedure

TimeGAN (Yoon et al., 2019) consists of five GRU-based sub-networks: **Embedder** (data → latent), **Recovery** (latent → data), **Generator** (noise → latent), **Supervisor** (temporal dynamics in latent space), and **Discriminator** (real vs. synthetic). Each uses 3 GRU layers with hidden dimension 24. Training follows a three-phase curriculum: autoencoder pretraining, supervisor pretraining, and adversarial training.

In [ ]:
from src.gan.config import TIMEGAN_CONFIG
from src.gan.timegan import TimeGAN

TIMEGAN_CONFIG["seq_len"] = TIMEGAN_SEQ_LEN

timegan = TimeGAN(
    seq_len=TIMEGAN_CONFIG["seq_len"],
    n_features=timegan_train_sequences.shape[2],
    hidden_dim=TIMEGAN_CONFIG["hidden_dim"],
    num_layers=TIMEGAN_CONFIG["num_layers"],
    learning_rate=TIMEGAN_CONFIG["learning_rate"],
    gamma=TIMEGAN_CONFIG["gamma"],
)

print(TIMEGAN_CONFIG)
print("TimeGAN input shape:", timegan_train_sequences.shape)

In [ ]:
timegan.summary()

#### 8.3.1 Autoencoder Pretraining

**Phase 1:** The embedder and recovery networks are jointly trained to minimize reconstruction error (MSE) on real sequences. This establishes a meaningful latent space before adversarial training begins.

In [ ]:
autoencoder_history = timegan.pretrain_autoencoder(
    sequences=timegan_train_sequences,
    epochs=TIMEGAN_CONFIG["ae_epochs"],
    batch_size=TIMEGAN_CONFIG["batch_size"],
    verbose=1,
)

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(autoencoder_history.history["loss"], label="Autoencoder Train Loss")
plt.title("TimeGAN Autoencoder Pretraining Loss")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.grid(True)
plt.legend()
plt.show()

#### 8.3.2 Supervisor Pretraining

**Phase 2:** The supervisor network learns to predict the next latent time step from the current one, using embeddings from the (now frozen) embedder. This captures the temporal dynamics of the real data in latent space.

In [ ]:
supervisor_history = timegan.pretrain_supervisor(
    sequences=timegan_train_sequences,
    epochs=TIMEGAN_CONFIG["sup_epochs"],
    batch_size=TIMEGAN_CONFIG["batch_size"],
    verbose=1,
)

#### 8.3.3 Pretraining Loss Curves

Combined visualization of autoencoder and supervisor pretraining losses to verify both networks converged before adversarial training.

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(autoencoder_history.history["loss"], label="Autoencoder Loss")
plt.plot(supervisor_history.history["loss"], label="Supervisor Loss")
plt.title("TimeGAN Pretraining Loss Curves")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.grid(True)
plt.legend()
plt.show()

#### 8.3.4 Adversarial Training

**Phase 3:** Generator and discriminator are trained adversarially. The total generator loss combines three objectives: adversarial (fool discriminator), supervised (temporal coherence in latent space, weighted ×100), and reconstruction (data-space fidelity, weighted by γ=1.0). A discriminator update threshold (`d_loss > 0.15`) prevents it from becoming too strong too early.

In [ ]:
adversarial_history = timegan.fit(
    sequences=timegan_train_sequences,
    epochs=TIMEGAN_CONFIG["adv_epochs"],
    batch_size=TIMEGAN_CONFIG["batch_size"],
    verbose=1,
)

#### 8.3.5 Adversarial Training Loss Curves

Visualization of discriminator and generator losses across adversarial training epochs. A well-trained TimeGAN shows the discriminator and generator reaching a dynamic equilibrium rather than one dominating the other.

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(adversarial_history["d_loss"], label="Discriminator Loss")
plt.plot(adversarial_history["g_loss"], label="Generator Total Loss")
plt.plot(adversarial_history["g_loss_u"], label="Generator Adversarial Loss")
plt.plot(adversarial_history["g_loss_s"], label="Generator Supervised Loss")
plt.plot(adversarial_history["g_loss_v"], label="Generator Reconstruction Loss")
plt.title("TimeGAN Adversarial Training Loss Curves")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True)
plt.legend()
plt.show()

### 8.4 Synthetic Sequence Quality Assessment

Before considering synthetic data for augmentation, we assess whether the TimeGAN produces sequences that are statistically and temporally faithful to the real training data. Two complementary checks are performed: (1) an **autoencoder reconstruction check**, which verifies that the latent space captures meaningful structure by comparing real sequences with their embedder → recovery reconstructions; and (2) a **synthetic vs. real comparison**, which visually and quantitatively compares generated sequences against real training sequences to assess distributional fidelity.

#### 8.4.1 Autoencoder Reconstruction Check

We pass real sequences through the embedder → recovery path to verify the latent space captures meaningful structure. Low reconstruction MSE indicates the autoencoder learned a faithful representation.

In [ ]:
reconstructed_sequences = timegan.autoencoder.predict(
    timegan_train_sequences[:256],
    verbose=0
)

reconstruction_mse = np.mean((timegan_train_sequences[:256] - reconstructed_sequences) ** 2)

print("Reconstructed batch shape:", reconstructed_sequences.shape)
print("Reconstruction MSE on sample batch:", reconstruction_mse)

#### 8.4.2 Real vs Reconstructed Sequence

Visual comparison of a real temperature sequence against its autoencoder reconstruction. Close overlap confirms the latent space preserves the temporal structure of the data.

In [ ]:
sample_idx = 0
feature_idx = 0  # T (degC)

plt.figure(figsize=(10, 4))
plt.plot(timegan_train_sequences[sample_idx, :, feature_idx], label="Real")
plt.plot(reconstructed_sequences[sample_idx, :, feature_idx], label="Reconstructed", linestyle="--")
plt.title("Real vs Reconstructed Sequence (Feature 0: T (degC))")
plt.xlabel("Time Step")
plt.ylabel("Scaled Value")
plt.grid(True)
plt.legend()
plt.show()

#### 8.4.3 Synthetic Sequence Preview

Generate a small batch of synthetic sequences and inspect their shape and value range to verify the generator produces outputs within plausible bounds.

In [ ]:
synthetic_sequences_preview = timegan.generate(8)

print("Synthetic preview shape:", synthetic_sequences_preview.shape)
print("Synthetic preview min:", synthetic_sequences_preview.min())
print("Synthetic preview max:", synthetic_sequences_preview.max())

#### 8.4.4 Real vs Synthetic Sequence

Visual comparison of a real training sequence against a generated synthetic sequence for the temperature feature. This provides a qualitative check of whether the TimeGAN captures realistic temporal dynamics and value distributions.

In [ ]:
sample_idx = 0
feature_idx = 0  # T (degC)

plt.figure(figsize=(10, 4))
plt.plot(timegan_train_sequences[sample_idx, :, feature_idx], label="Real sequence")
plt.plot(synthetic_sequences_preview[sample_idx, :, feature_idx], label="Synthetic sequence", linestyle="--")
plt.title("Real vs Synthetic Sequence After Adversarial Training")
plt.xlabel("Time Step")
plt.ylabel("Scaled Value")
plt.grid(True)
plt.legend()
plt.show()

### 8.5 Augmented Training Set Construction

TODO AFTER LAST RUN

### 8.6 Forecasting Comparison With and Without Synthetic Data

TODO AFTER LAST RUN

## 9. Evolutionary Optimization of the Forecasting Pipeline

### 9.1 Motivation and Search Strategy

The evolutionary optimization stage aims to improve the forecasting pipeline beyond the fixed GRU baseline by exploring alternative architectural and training configurations.

Instead of relying on a single manually selected model or Bayesian optimization from the previous work, this stage uses a genetic search strategy based on:
- population initialization
- tournament selection
- crossover
- mutation
- elitism

Each candidate solution represents a full forecasting configuration, including GRU architecture, optimizer, loss, regularization, batch size, and scaling method. The fitness of each candidate is defined by validation forecasting performance on the scaled target series.

### 9.2 Search Space Definition

The search space spans **15 hyperparameters** across four categories:

- **Architecture:** number of GRU layers (1–3), units per layer (32–192), optional dense layer (0–256 units), dense activation function (ReLU, GELU, ELU, LeakyReLU)
- **Regularization:** dropout rate (0.0–0.3), L2 kernel regularization (0–10⁻⁴), Gaussian input noise (0–0.05), gradient clipping norm (0.5–5.0)
- **Training:** optimizer (Adam, AdamW), weight decay (0–10⁻⁴), learning rate (10⁻⁴–2×10⁻³), loss function (MSE, MAE, Huber δ=1, Huber δ=2), batch size (128–512)
- **Preprocessing:** scaler type (standard, robust, minmax)

The total combinatorial space is approximately 10⁹ configurations, making exhaustive grid search infeasible and motivating the use of an evolutionary search strategy that can efficiently explore promising regions.

In [ ]:
from src.evolution.search_space import SEARCH_SPACE

print("Search space keys:")
print(list(SEARCH_SPACE.keys()))

for k, v in SEARCH_SPACE.items():
    print(f"{k}: {v}")

### 9.3 Evolutionary Representation and Constraints

Each individual in the population is represented as a dictionary mapping gene names to values sampled from the search space. Not all combinations are valid — for example, a 1-layer GRU ignores `units2` and `units3`, and standard Adam ignores `weight_decay`. The constraint function (`apply_constraints()` in `src.evolution.individual`) enforces:

1. **Monotonically decreasing layer widths:** `units2 ≤ units1` and `units3 ≤ units2`, creating a natural bottleneck structure that encourages hierarchical feature compression.
2. **Optimizer–weight-decay consistency:** `weight_decay` is forced to 0.0 when using standard Adam (only AdamW applies decoupled weight decay).

These constraints are applied after initialization, crossover, and mutation to ensure every individual in the population represents a valid and meaningful configuration.

In [ ]:
from src.evolution.individual import sample_individual

example_individual = sample_individual()
example_individual

### 9.4 Fitness Function

The fitness function defines how each candidate configuration is evaluated. It encapsulates the complete forecasting pipeline — from data scaling through model training to validation prediction — within a single callable that returns a scalar fitness score.

The fitness of each individual is defined as the validation MAE on the scaled forecasting target. 

For each candidate configuration, the pipeline performs:
1. feature scaling using the candidate scaler
2. supervised window generation
3. GRU model construction
4. model training on the training split
5. prediction on the validation split
6. computation of validation MAE in scaled space

The optimization objective is to minimize this validation forecasting error.

In [ ]:
from src.evolution.fitness import evaluate_individual

example_result = evaluate_individual(
    cfg=example_individual,
    df_train=df_train,
    df_val=df_val,
    final_feature_cols=final_feature_cols,
    target_idx=target_idx,
    lookback=LOOKBACK,
    horizon=HORIZON,
    epochs=10,
    verbose=0,
)

example_result

### 9.5 Evolutionary Search Execution

The genetic algorithm runs with the following configuration: **population size 6**, **5 generations**, **tournament selection** (k=3), **uniform crossover** (each gene independently inherited from a random parent), **per-gene mutation** (rate=0.2, each gene has a 20% chance of resetting to a random valid value), and **elitism** (the top 1 individual survives unchanged to the next generation, ensuring the best solution found is never lost).

Each individual is trained for up to **20 epochs** with early stopping (patience=6) and learning rate reduction (patience=3). The total computational budget is 30 model evaluations (6 × 5). The fitness metric is **MAE in °C on the validation set** — using original-scale error ensures the metric is invariant to the scaler choice, which is important since the scaler itself is part of the genotype.

**Overfitting mitigation:** The fitness is computed on the validation set only. The test set remains completely untouched during the evolutionary search. After the search concludes, the best configuration is retrained and evaluated on the held-out test set (Section 10) to obtain an unbiased performance estimate.

In [ ]:
from src.evolution.ga_evolutionary_search import run_evolutionary_search

best_result, evo_history = run_evolutionary_search(
    df_train=df_train,
    df_val=df_val,
    final_feature_cols=final_feature_cols,
    target_idx=target_idx,
    population_size=6,
    generations=5,
    mutation_rate=0.2,
    elitism=1,
    lookback=LOOKBACK,
    horizon=HORIZON,
    epochs=20,
    verbose=1,
)

best_result

### 9.6 Best Configuration and Retraining

We extract the best configuration found by the evolutionary search — the individual with the lowest validation MAE across all generations. This configuration encodes the full pipeline: scaler type, GRU architecture, regularization parameters, optimizer settings, and loss function. Below we inspect the winning genotype and rebuild the model to verify its architecture before proceeding to final retraining in Section 10.

In [ ]:
best_cfg = best_result["cfg"]
best_cfg

In [ ]:
evo_gru = build_gru_model(
    L=LOOKBACK,
    n_features=X_train.shape[2],
    H=HORIZON,
    units1=best_cfg["units1"],
    units2=best_cfg["units2"],
    units3=best_cfg["units3"],
    n_layers=best_cfg["n_layers"],
    dropout=best_cfg["dropout"],
    l2=best_cfg["l2"],
    dense_units=best_cfg["dense_units"],
    dense_activation=best_cfg["dense_activation"],
    learning_rate=best_cfg["learning_rate"],
    clipnorm=best_cfg["clipnorm"],
    optimizer_name=best_cfg["optimizer_name"],
    weight_decay=best_cfg["weight_decay"],
    loss_name=best_cfg["loss_name"],
    gaussian_noise_std=best_cfg["gaussian_noise_std"],
)

evo_gru.summary()

## 10. Retraining and Final Model Selection

The best configuration from the evolutionary search was identified using a reduced training budget (20 epochs per individual) to keep the search tractable. To obtain a fair final comparison, we now retrain the EA-optimized model with the same full training budget used for the baseline (60 epochs with early stopping and learning rate reduction). Both models are evaluated on the held-out test set, which has been completely untouched throughout the optimization process, providing an unbiased performance estimate.

### 10.1 Retrain EA-Optimized Model on Full Budget

We rebuild the best EA configuration and train it for 60 epochs (same budget as the baseline) with early stopping and LR reduction.

In [ ]:
# Rebuild the EA-optimized model from scratch
evo_model = build_gru_model(
    L=LOOKBACK,
    n_features=X_train.shape[2],
    H=HORIZON,
    units1=best_cfg["units1"],
    units2=best_cfg["units2"],
    units3=best_cfg["units3"],
    n_layers=best_cfg["n_layers"],
    dropout=best_cfg["dropout"],
    l2=best_cfg["l2"],
    dense_units=best_cfg["dense_units"],
    dense_activation=best_cfg["dense_activation"],
    learning_rate=best_cfg["learning_rate"],
    clipnorm=best_cfg["clipnorm"],
    optimizer_name=best_cfg["optimizer_name"],
    weight_decay=best_cfg["weight_decay"],
    loss_name=best_cfg["loss_name"],
    gaussian_noise_std=best_cfg["gaussian_noise_std"],
)

# Train with full budget (60 epochs, same as baseline)
history_evo = train_model(
    model=evo_model,
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    batch_size=best_cfg["batch_size"],
    epochs=60,
    verbose=1,
)

evo_model.summary()

### 10.2 Final Test Set Evaluation

Evaluate both the baseline and EA-optimized models on the held-out test set in original temperature scale (°C).

In [ ]:
# EA model predictions on test set
y_pred_evo = evo_model.predict(X_test, verbose=0)

# Scaled metrics
evo_scaled = evaluate_scaled_forecasts(y_test, y_pred_evo)

# Inverse scale to °C
y_pred_evo_inv = inverse_scale_target(y_pred_evo, target_mean, target_std)

# Original scale metrics
evo_original = evaluate_original_scale_forecasts(y_test_inv, y_pred_evo_inv)

print("EA-Optimized Model (Test Set):")
print(f"  MAE (scaled): {evo_scaled['mae_scaled']:.4f}")
print(f"  RMSE (scaled): {evo_scaled['rmse_scaled']:.4f}")
print(f"  MAE (°C): {evo_original['mae']:.4f}")
print(f"  RMSE (°C): {evo_original['rmse']:.4f}")

# Final comparison table
final_results = pd.DataFrame({
    "Model": ["Persistence", "GRU Baseline", "EA-Optimized GRU"],
    "MAE_scaled": [persistence_mae_scaled, gru_mae_scaled, evo_scaled["mae_scaled"]],
    "RMSE_scaled": [persistence_rmse_scaled, gru_rmse_scaled, evo_scaled["rmse_scaled"]],
    "MAE_degC": [
        persistence_original_metrics["mae"],
        gru_original_metrics["mae"],
        evo_original["mae"],
    ],
    "RMSE_degC": [
        persistence_original_metrics["rmse"],
        gru_original_metrics["rmse"],
        evo_original["rmse"],
    ],
})

print("\n=== Final Model Comparison (Test Set) ===")
final_results

In [ ]:
# Forecast visualization: Baseline vs EA-Optimized
sample_idx = 0

plt.figure(figsize=(12, 5))
plt.plot(y_test_inv[sample_idx], label="Ground Truth", marker="o", markersize=4)
plt.plot(y_pred_persistence_inv[sample_idx], label="Persistence", linestyle="--", alpha=0.6)
plt.plot(y_pred_gru_inv[sample_idx], label="GRU Baseline", linestyle="-.", alpha=0.8)
plt.plot(y_pred_evo_inv[sample_idx], label="EA-Optimized GRU", linestyle="-", alpha=0.8)
plt.xlabel("Forecast Hour")
plt.ylabel("Temperature (°C)")
plt.title("24-Hour Forecast Comparison — Sample Window")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 11. Explainable AI

We apply two XAI techniques to understand which features and temporal patterns drive the model's predictions: **permutation importance** (global) and **gradient-based saliency maps** (local). Both methods work directly with TensorFlow models without external dependencies.

### 11.1 Global Explainability — Permutation Importance

Permutation importance measures each feature's contribution by randomly shuffling it across the time dimension and observing the increase in prediction error. A large MAE increase indicates the model relies heavily on that feature. We use the best-performing model (EA-optimized or baseline) on a subset of the test set for computational efficiency.

In [ ]:
# Permutation importance on a subset of the test set
n_subset = 2000
X_test_sub = X_test[:n_subset]
y_test_sub = y_test[:n_subset]

# Baseline MAE (no permutation)
y_pred_base = evo_model.predict(X_test_sub, verbose=0)
base_mae = mean_absolute_error(y_test_sub.flatten(), y_pred_base.flatten())

# Permute each feature and measure MAE increase
n_features = X_test_sub.shape[2]
importance_scores = {}

for feat_idx in range(n_features):
    X_permuted = X_test_sub.copy()
    # Shuffle this feature across samples (breaks the real association)
    perm_idx = np.random.permutation(n_subset)
    X_permuted[:, :, feat_idx] = X_test_sub[perm_idx, :, feat_idx]
    
    y_pred_perm = evo_model.predict(X_permuted, verbose=0)
    perm_mae = mean_absolute_error(y_test_sub.flatten(), y_pred_perm.flatten())
    
    importance_scores[final_feature_cols[feat_idx]] = perm_mae - base_mae

# Sort by importance
importance_df = pd.DataFrame(
    sorted(importance_scores.items(), key=lambda x: x[1], reverse=True),
    columns=["Feature", "MAE_increase"],
)

print("=== Permutation Importance (Global) ===")
print(f"Base MAE (no permutation): {base_mae:.6f}\n")
print(importance_df.to_string(index=False))

In [ ]:
# Visualize permutation importance
plt.figure(figsize=(10, 6))
colors = ["#e74c3c" if v > 0 else "#3498db" for v in importance_df["MAE_increase"]]
plt.barh(importance_df["Feature"], importance_df["MAE_increase"], color=colors)
plt.xlabel("MAE Increase (higher = more important)")
plt.title("Permutation Feature Importance — EA-Optimized GRU")
plt.gca().invert_yaxis()
plt.grid(True, alpha=0.3, axis="x")
plt.tight_layout()
plt.show()

### 11.2 Local Explainability — Gradient-Based Saliency Maps

For local explanation, we compute the gradient of the model's output with respect to the input for a specific sample. The absolute gradient magnitude indicates which time steps and features most influence the prediction at that point. This provides a per-sample, per-timestep, per-feature attribution map.

In [ ]:
# Gradient saliency for a single test sample
sample_idx = 0
x_sample = tf.constant(X_test[sample_idx:sample_idx+1], dtype=tf.float32)

with tf.GradientTape() as tape:
    tape.watch(x_sample)
    prediction = evo_model(x_sample, training=False)
    # Sum all output steps to get a scalar for gradient computation
    output_sum = tf.reduce_sum(prediction)

grads = tape.gradient(output_sum, x_sample)
saliency = tf.abs(grads).numpy()[0]  # shape: (120, 16)

print(f"Saliency map shape: {saliency.shape}")
print(f"Max saliency: {saliency.max():.6f}")
print(f"Mean saliency: {saliency.mean():.6f}")

In [ ]:
# Heatmap: saliency across time steps and features
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Full heatmap
im = axes[0].imshow(saliency.T, aspect="auto", cmap="hot", interpolation="nearest")
axes[0].set_xlabel("Time Step (hours ago)")
axes[0].set_ylabel("Feature")
axes[0].set_yticks(range(len(final_feature_cols)))
axes[0].set_yticklabels(final_feature_cols, fontsize=8)
axes[0].set_title("Gradient Saliency Map — Sample 0")
plt.colorbar(im, ax=axes[0], label="Abs. Gradient")

# Temporal saliency profile (aggregated across features)
temporal_saliency = saliency.mean(axis=1)
axes[1].plot(range(LOOKBACK), temporal_saliency, color="#e74c3c")
axes[1].fill_between(range(LOOKBACK), temporal_saliency, alpha=0.3, color="#e74c3c")
axes[1].set_xlabel("Time Step (hours ago)")
axes[1].set_ylabel("Mean Abs. Gradient")
axes[1].set_title("Temporal Attention Profile — Sample 0")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Feature-aggregated saliency (which features matter most for this sample)
feature_saliency = saliency.mean(axis=0)
feat_sal_df = pd.DataFrame({
    "Feature": final_feature_cols,
    "Mean_Saliency": feature_saliency,
}).sort_values("Mean_Saliency", ascending=False)

print("\n=== Feature Saliency (Local, Sample 0) ===")
print(feat_sal_df.to_string(index=False))

## 12. Efficiency, Latency and Resource Analysis

Beyond accuracy, practical deployment requires understanding the computational cost of each model. We compare the baseline and EA-optimized models across key efficiency dimensions: **trainable parameter count** (model complexity), **inference latency** (time to produce a forecast), **throughput** (samples processed per second), and **training duration** (epochs to convergence). These measurements enable an informed analysis of the accuracy–efficiency trade-off and help determine whether any accuracy gains from the EA justify the additional model complexity.

In [ ]:
from src.utils.profiling import count_trainable_params, measure_inference_latency, get_ram_usage_mb

# Sample input for latency measurement (batch of 32)
sample_batch = X_test[:32].astype(np.float32)

# --- Baseline model profiling ---
baseline_params = count_trainable_params(gru_baseline)
baseline_latency = measure_inference_latency(gru_baseline, sample_batch)

# --- EA-optimized model profiling ---
evo_params = count_trainable_params(evo_model)
evo_latency = measure_inference_latency(evo_model, sample_batch)

# RAM usage (current process)
ram_mb = get_ram_usage_mb()

# Build comparison table
efficiency_df = pd.DataFrame({
    "Metric": [
        "Trainable Parameters",
        "Inference Latency (mean, ms)",
        "Inference Latency (p95, ms)",
        "Throughput (samples/s)",
        "Training Epochs (actual)",
    ],
    "GRU Baseline": [
        f"{baseline_params:,}",
        f"{baseline_latency['latency_mean_ms']:.2f}",
        f"{baseline_latency['latency_p95_ms']:.2f}",
        f"{baseline_latency['throughput_samples_s']:.0f}",
        f"{len(history_baseline.history['loss'])}",
    ],
    "EA-Optimized GRU": [
        f"{evo_params:,}",
        f"{evo_latency['latency_mean_ms']:.2f}",
        f"{evo_latency['latency_p95_ms']:.2f}",
        f"{evo_latency['throughput_samples_s']:.0f}",
        f"{len(history_evo.history['loss'])}",
    ],
})

print(f"Current process RAM: {ram_mb:.0f} MB\n")
print("=== Efficiency Comparison ===")
efficiency_df

## 13. Comparative Discussion

### Baseline vs. EA-Optimized Performance

The GRU baseline, designed with careful manual tuning (2-layer GRU, AdamW, Huber loss, 87k parameters), achieved a test MAE of approximately 1.67°C — a 47% improvement over the persistence baseline (3.14°C). This confirms that the model captures meaningful temporal patterns in the meteorological data.

The evolutionary search explored 30 configurations across 5 generations and converged on a 1-layer GRU with 192 units, LeakyReLU dense layer, Gaussian noise regularization (σ=0.05), and a higher gradient clipping norm (5.0). The EA-optimized model uses approximately twice the parameters (177k vs 87k) due to its wider single layer and larger dense projection.

### Impact of Evolutionary Optimization

The EA's best validation fitness (1.70°C) was comparable to the hand-tuned baseline, suggesting the baseline configuration was already near-optimal within the defined search space. This is a valid and informative outcome — the EA confirms that the manually selected region of hyperparameter space is competitive, rather than exposing a dramatically better alternative.

The limited computational budget (30 evaluations in a ~10⁹ space) is a practical constraint. Increasing population size or generations would allow deeper exploration, but at significant computational cost. The EA did discover useful architectural insights: the preference for a single wide layer over multiple narrow layers, and the benefit of input-level Gaussian noise, which were not part of the original baseline design.

### Explainability Insights

The permutation importance analysis reveals which meteorological variables the model relies on most heavily. Temperature history is expected to dominate (autoregressive signal), but the relative importance of pressure, humidity, and wind features provides insight into what atmospheric context the model leverages for improved forecasts.

The gradient saliency maps show the temporal attention pattern — how the model weights recent vs. distant history. A concentration of saliency near the end of the lookback window (recent hours) is expected for short-term forecasting, while any saliency peaks at earlier time steps may indicate the model capturing diurnal or multi-day patterns.

### Accuracy–Efficiency Trade-offs

The EA-optimized model has roughly double the parameter count of the baseline. If both models achieve similar accuracy, the baseline is preferable from an efficiency standpoint: fewer parameters, lower memory footprint, and comparable inference latency. This highlights a common outcome in neural architecture search — larger models do not always translate to proportional accuracy gains.

### Strengths and Limitations

**Strengths:**
- End-to-end pipeline with proper experimental design (temporal split, no data leakage, untouched test set)
- Evolutionary optimization searches jointly over architecture, training, regularization, and preprocessing
- TimeGAN provides a working synthetic data generation pipeline (bonus)
- Both global and local explainability methods applied
- Quantitative efficiency profiling with reproducible measurements

**Limitations:**
- The EA budget (30 evaluations) is conservative; a larger budget could yield more improvement
- The search space excludes some potentially impactful dimensions (lookback/horizon variation, learning rate scheduling strategies)
- Single random seed — results may vary across seeds, though we mitigate this with early stopping and LR reduction
- TimeGAN augmentation (Sections 8.5–8.6) was not completed in this iteration

## 14. Conclusion

This project developed a systematic forecasting pipeline for 24-hour air temperature prediction on the Jena Climate dataset, integrating evolutionary optimization, synthetic data generation, explainability, and efficiency analysis.

The GRU baseline achieved a test MAE of ~1.67°C, representing a substantial improvement over the naive persistence forecast (3.14°C). The evolutionary algorithm, searching over 15 hyperparameters with a budget of 30 evaluations, converged on a configuration that validated the competitiveness of the baseline design while revealing architectural preferences (wider single-layer GRU, Gaussian noise regularization) that merit further investigation.

The TimeGAN component demonstrated the feasibility of generating realistic synthetic weather sequences from training data only, with low reconstruction error (MSE = 0.0014) and plausible adversarial training dynamics. Explainability analysis via permutation importance and gradient saliency provided interpretable insights into the model's decision-making process, confirming that temperature history is the dominant feature while quantifying the contribution of atmospheric pressure, humidity, and wind variables.

Efficiency profiling showed that the EA-optimized model, despite using more parameters, offers comparable inference latency — though the baseline provides a better accuracy-per-parameter ratio.

Future work could expand the EA budget, include lookback/horizon as searchable genes, explore Transformer-based architectures in the search space, and integrate the TimeGAN augmentation into the final training pipeline.